In [17]:
import pandas as pd

df = pd.read_excel('/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Dutch_dataset.xlsx', sheet_name='output')
df = df.dropna(how='all')

df.head(10)

,id,Name,Gender,Mom,Age,Video,Upload date
0,1,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=1AZUFJSEags,NaN
1,2,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=XkneWNRXhPo,NaN
2,3,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=qBlvkXycXiE,NaN
3,4,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=u4L4mrnS-O4,NaN
4,5,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=1fVA1wWSzU4,NaN
5,6,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=7buKrBom1H4,NaN
6,7,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=QAG9kOaOFak,NaN
7,8,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=L2MVe0HXvr0,NaN
8,9,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=34sIWMnBQ8w,NaN
9,10,Jasmijn Oonk,f,n,25,https://www.youtube.com/watch?v=YTyALr8ihro,NaN


In [18]:
import os 

input_folder = "Dutch Transcripts"
output_folder = "Transcript"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def get_transcript_content(file_id):
    file_name = f"{file_id + 1}_anonymized.txt"
    file_path = os.path.join(input_folder, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            output_path = os.path.join(output_folder, file_name)
            with open(output_path, 'w', encoding='utf-8') as output_file:
                output_file.write(content)
            return content
    except FileNotFoundError:
        print(f"File not found: {file_name}")
        return None

df['Transcript'] = df['id'].apply(get_transcript_content)


df = df[["id", "Transcript"]]
display(df)

File not found: 121_anonymized.txt


,id,Transcript
0,1,"<PERSON> heeft een nieuwe fles, de stalen fles..."
1,2,maar liefst duizend euro krijgt van de overhei...
2,3,we leven in een rare en moeilijke tijd. in de ...
3,4,valt ik op vrouwen? ik heb heel lang getwijfel...
4,5,"wie is daar? kom voor de houstour. oké, ik kom..."
...,...,...
115,116,"nou ben ik samen met felix, die jullie natuurl..."
116,117,artie is uitlogeren en <NRP> sfeer te zoeken h...
117,118,"ik heb die ingekocht, die, die, die heb ik wee..."
118,119,"ja, dus vandaar dat ik dit ook gewoon echt met..."


In [19]:
import pandas as pd
from langdetect import detect

detected_language = detect(' '.join(df['Transcript'].astype(str)))

if detected_language == 'en':
    new_column_name = 'Transcript_english'
elif detected_language == 'es':
    new_column_name = 'Transcript_spanish'
elif detected_language == 'nl':
    new_column_name = 'Transcript_dutch'
elif detected_language == 'tr':
    new_column_name = 'Transcript_turkish'
else:
    new_column_name = f'Transcript_{detected_language}'

df.rename(columns={'Transcript': new_column_name}, inplace=True)

In [9]:
pip install --upgrade spacy "pydantic>=1.9.0,<2.0"

DEPRECATION: Loading egg at /opt/homebrew/lib/python3.11/site-packages/jupyter-1.0.0-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
  Using cached pydantic-1.10.19-cp311-cp311-macosx_11_0_arm64.whl.metadata (152 kB)
  Using cached thinc-8.3.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (15 kB)
Using cached pydantic-1.10.19-cp311-cp311-macosx_11_0_arm64.whl (2.3 MB)
Using cached thinc-8.3.2-cp311-cp311-macosx_11_0_arm64.whl (774 kB)
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.8.2
    Uninstalling pydantic-1.8.2:
      Successfully uninstalled pydantic-1.8.2
  Attempting uninstall: thinc
    Found existing installation: thinc 9.1.1
    Uninstalling thinc-9.1.1:
      Successfully uninstalled thinc-9.1.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [20]:
import re
import os
import spacy
import pandas as pd

transcript_column = [col for col in df.columns if col.startswith('Transcript_')][0]
detected_language = transcript_column.split('_')[1]

if detected_language == 'english':
    nlp = spacy.load("en_core_web_sm")
elif detected_language == 'spanish':
    nlp = spacy.load("es_core_news_sm")
elif detected_language == 'dutch':
    nlp = spacy.load("nl_core_news_sm")
else:
    raise ValueError(f"spaCy model is not available for the detected language: {detected_language}")

# Function to clean and remove text between brackets 
def remove_text_in_brackets(text):
    if isinstance(text, str):
        return re.sub(r'[<\[\(\{\}\)\]>]', '', text)
    else:
        return ''

df['clean_transcript'] = df[transcript_column].apply(remove_text_in_brackets)

# Tokenization and text analysis
def tokenize_and_pos_tag(text):
    doc = nlp(text) 
    tokens = [token.text for token in doc]
    pos_tags = [(token.text, token.pos_) for token in doc]
    
    if 'lemmatizer' in nlp.pipe_names:
        lemma = [(token.text, token.lemma_) for token in doc]
    else:
        lemma = [(token.text, None) for token in doc]
    
    entity = [(ent.text, ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
    
    return tokens, pos_tags, lemma, entity

df[['tokens', 'pos_tag', 'lemma', 'entity']] = df['clean_transcript'].apply(lambda text: pd.Series(tokenize_and_pos_tag(text)))

def split_into_phrases(text):
    if isinstance(text, str):
        doc = nlp(text)
        return [sent.text.strip() for sent in doc.sents]
    return []

df['phrases'] = df['clean_transcript'].apply(split_into_phrases)
display(df)

,id,Transcript_dutch,clean_transcript,tokens,pos_tag,lemma,entity,phrases
0,1,"<PERSON> heeft een nieuwe fles, de stalen fles...","PERSON heeft een nieuwe fles, de stalen fles, ...","[PERSON, heeft, een, nieuwe, fles, ,, de, stal...","[(PERSON, PROPN), (heeft, VERB), (een, DET), (...","[(PERSON, PERSON), (heeft, hebben), (een, een)...","[(PERSON, 0, 6, ORG), (5, 295, 296, CARDINAL),...","[PERSON heeft een nieuwe fles, de stalen fles,..."
1,2,maar liefst duizend euro krijgt van de overhei...,maar liefst duizend euro krijgt van de overhei...,"[maar, liefst, duizend, euro, krijgt, van, de,...","[(maar, ADV), (liefst, ADV), (duizend, NUM), (...","[(maar, maar), (liefst, liefst), (duizend, dui...","[(duizend euro, 12, 24, DATE), (jou, 337, 340,...",[maar liefst duizend euro krijgt van de overhe...
2,3,we leven in een rare en moeilijke tijd. in de ...,we leven in een rare en moeilijke tijd. in de ...,"[we, leven, in, een, rare, en, moeilijke, tijd...","[(we, PRON), (leven, VERB), (in, ADP), (een, D...","[(we, we), (leven, leven), (in, in), (een, een...","[(LOCATION, 233, 241, ORG), (LOCATION, 370, 37...","[we leven in een rare en moeilijke tijd., in d..."
3,4,valt ik op vrouwen? ik heb heel lang getwijfel...,valt ik op vrouwen? ik heb heel lang getwijfel...,"[valt, ik, op, vrouwen, ?, ik, heb, heel, lang...","[(valt, VERB), (ik, PRON), (op, ADP), (vrouwen...","[(valt, vallen), (ik, ik), (op, op), (vrouwen,...","[(eerste, 798, 804, ORDINAL), (tweede, 956, 96...","[valt ik op vrouwen?, ik heb heel lang getwijf..."
4,5,"wie is daar? kom voor de houstour. oké, ik kom...","wie is daar? kom voor de houstour. oké, ik kom...","[wie, is, daar, ?, kom, voor, de, houstour, .,...","[(wie, PRON), (is, AUX), (daar, ADV), (?, PUNC...","[(wie, wie), (is, zijn), (daar, daar), (?, ?),...","[(woonkamer, 98, 107, ORG), (drie, 159, 163, C...","[wie is daar? kom voor de houstour., oké, ik k..."
...,...,...,...,...,...,...,...,...
115,116,"nou ben ik samen met felix, die jullie natuurl...","nou ben ik samen met felix, die jullie natuurl...","[nou, ben, ik, samen, met, felix, ,, die, jull...","[(nou, ADV), (ben, AUX), (ik, PRON), (samen, A...","[(nou, nou), (ben, zijn), (ik, ik), (samen, sa...","[(PERSON, 135, 141, ORG), (nacht, 180, 185, PE...","[nou ben ik samen met felix, die jullie natuur..."
116,117,artie is uitlogeren en <NRP> sfeer te zoeken h...,artie is uitlogeren en NRP sfeer te zoeken hie...,"[artie, is, uitlogeren, en, NRP, sfeer, te, zo...","[(artie, NOUN), (is, AUX), (uitlogeren, VERB),...","[(artie, artie), (is, zijn), (uitlogeren, uitl...","[(NRP, 23, 26, ORG), (DATE_TIME, 332, 341, WOR...",[artie is uitlogeren en NRP sfeer te zoeken hi...
117,118,"ik heb die ingekocht, die, die, die heb ik wee...","ik heb die ingekocht, die, die, die heb ik wee...","[ik, heb, die, ingekocht, ,, die, ,, die, ,, d...","[(ik, PRON), (heb, AUX), (die, DET), (ingekoch...","[(ik, ik), (heb, hebben), (die, die), (ingekoc...","[(DATE_TIME, 363, 372, WORK_OF_ART), (één, 397...","[ik heb die ingekocht, die, die, die heb ik we..."
118,119,"ja, dus vandaar dat ik dit ook gewoon echt met...","ja, dus vandaar dat ik dit ook gewoon echt met...","[ja, ,, dus, vandaar, dat, ik, dit, ook, gewoo...","[(ja, INTJ), (,, PUNCT), (dus, CCONJ), (vandaa...","[(ja, ja), (,, ,), (dus, dus), (vandaar, vanda...","[(LOCATION, 613, 621, ORG), (DATE_TIME, 782, 7...","[ja, dus vandaar dat ik dit ook gewoon echt me..."


In [21]:

mental_health_words_nl = """Droevig, Gelukkig, Hopeloos, Hoopvol, Leeg, Tevreden, Ontevreden, 
Laag, Hoog, Verdoofd, Responsief, Melancholisch, Vrolijk, Verslagen, Succesvol, Verdriet, 
Aangenaam, Blauw, Optimistisch, Somber, Helder, Ontmoedigd, Geïnspireerd, Vreugdeloos, Blij, 
Ongevoelig, Gevoelig, Onverschillig, Zorgzaam, Ongemotiveerd, Gemotiveerd, Niet betrokken, 
Bevlogen, Verveeld, Geïnteresseerd, Afstandelijk, Betrokken, Onvoldaan, Vervuld, Levenloos, 
Levendig, Ontkoppeld, Verbonden, Blanco, Expressief, Emotieloos, Emotioneel, Schuldig, Onschuldig, 
Waardeloos, Waardevol, Beschaamd, Trots, Mislukking, Succes, Ongeschikt, Bekwaam, Zelfverwijt,
Zelfacceptatie, Berouwvol, Tevreden, Nuttig, Gebrekkig, Perfect, Schaamte, Assertief, Onwaardig,
Waardig, Onverdienstelijk, Verdienstelijk, Moe, Energiek, Uitgeput, Opgefrist, Afgetapt, 
Opgeladen, Vermoeid, Uitgerust, Slaperig, Actief, Energiek, Levendig, Slaperig, Alert, 
Rundown, Gerevitaliseerd, Opgebrand, Vernieuwd, Angstig, Kalm, Zenuwachtig, Zelfverzekerd, 
Rusteloos, Zenuwachtig, Samengesteld, Angstig, Dapper, Gespannen, Vredig, Bezorgd, Verzekerd, 
Onrustig, Paniek, Chill, Onrustig, Verontrust, Onrustig, Op je gemak, Afgeleid, Aandachtig, 
Vergeetachtig, Mindful, Besluiteloos, Besluitvaardig, Ongericht, Gefocust, Helder, Mistig, 
Lucide, Verspreid, Georganiseerd, Verward, Zeker, Verloren, Grondgebonden, Wazig, Scherp, 
Afwezig-geestig, Wakker, Hoofdpijn, Opluchting, Misselijkheid, Spierpijn, Flexibiliteit, 
Pijn, Gemak, Pijn, Zwak, Sterk, Duizelig, Vast, Pijn, Strak, Los, Lichaamspijn, Welbevinden, 
Slapeloosheid, Slaperig, Uitslapen, Overslapen, Verstoord, Ononderbroken, Nachtmerrie, 
Gebroken, Slapen, Slaaptekort, Welterusten, Inappetent, Eetlust, Cravings, Tevredenheid, 
Eetbuien, Voedzaam, Ondergewicht, Gezond, Evenwichtig, Ondervoed, Geïsoleerd, Eenzaam, 
Samen, Teruggetrokken, Afgewezen, Geaccepteerd, Onbemind, Geliefd, Onbegrepen, Onbegrepen,
Verwaarloosd, Verzorgd, Vermeden, Omarmd, Zinloos, Kostbaar, Significant, Somber, Zinloos,
Zinvol, Vergeefs, Vruchtbaar, Compleet, Prestatie, Vertrouwen, Pessimistisch, Aangemoedigd,
Gedoemd, Veerkrachtig, Doelloos, Gedreven, Onzeker, Zelfverzekerd, Incompetent, Competent"""


mental_health_words_nl = [word.strip().lower() for word in mental_health_words_nl.split(',')]

In [22]:
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span

# Inicializar Spacy y el PhraseMatcher
nlp = spacy.load("nl_core_news_sm")
matcher = PhraseMatcher(nlp.vocab)
patterns = [nlp.make_doc(word) for word in mental_health_words_nl]
matcher.add("MENTAL_HEALTH", patterns)

# Función para detectar las entidades relacionadas con salud mental
def detect_mental_health_entities(doc):
    matches = matcher(doc)
    spans = [Span(doc, start, end, label="MENTAL_HEALTH") for match_id, start, end in matches]
    spans = spacy.util.filter_spans(spans)
    existing_ents = list(doc.ents)
    non_conflicting_spans = [span for span in spans if not any(ent.start < span.end and span.start < ent.end for ent in existing_ents)]
    doc.ents = existing_ents + non_conflicting_spans

    mental_health_entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ == "MENTAL_HEALTH"]
    return mental_health_entities

df['mental_health_entities'] = df['clean_transcript'].apply(
    lambda text: detect_mental_health_entities(nlp(text))
)

output_file = '/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Dutch/Dutch_Dataset_Processed.csv'
os.makedirs(os.path.dirname(output_file), exist_ok=True)  # Ensure the directory exists
df.to_csv(output_file, index=False)

print(f"File has been saved as: {output_file}")

File has been saved as: /Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Dutch/Dutch_Dataset_Processed.csv


In [15]:
FPS = ["ik", "mij", "me", "mijn", "met mij"]
FPP = ["wij", "we", "ons", "onze", "met ons"]
SPS = ["jij", "je", "jou", "uw", "u", "met jou", "met u"]
SPP = ["jullie", "u", "met jullie"]
TPS = ["hij", "hem", "zijn", "met hem", "zij", "haar", "met haar", "het"]
TPP = ["zij", "ze", "hen", "hun", "met hen"]

pronouns_dict = {
    "FPS": FPS,
    "FPP": FPP,
    "SPS": SPS,
    "SPP": SPP,
    "TPS": TPS,
    "TPP": TPP,
}

results = []

for index, row in df.iterrows():
    row_data = {"clean_transcript": row["clean_transcript"]}
    for category, pronouns in pronouns_dict.items():
        counts = {p: 0 for p in pronouns}
        pronoun_list = []
        for token, tag in zip(row['tokens'], row['pos_tag']):
            if "PRON" in tag and token.lower() in pronouns:
                counts[token.lower()] += 1
                pronoun_list.append(token)

        row_data[f"{category.lower()}_counts"] = [(k, v) for k, v in counts.items()]
        row_data[category] = sum(counts.values())

    results.append(row_data)

df_pronouns = pd.DataFrame(results)


display(df_pronouns)
output_file_2 = '/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Dutch/Dutch_Pronouns_Dataset.csv'
os.makedirs(os.path.dirname(output_file_2), exist_ok=True)  # Ensure the directory exists
df_pronouns.to_csv(output_file_2, index=False)

print(f"File has been saved as: {output_file_2}")

,clean_transcript,fps_counts,FPS,fpp_counts,FPP,sps_counts,SPS,spp_counts,SPP,tps_counts,TPS,tpp_counts,TPP
0,"PERSON heeft een nieuwe fles, de stalen fles, ...","[(ik, 9), (mij, 1), (me, 1), (mijn, 2), (met m...",13,"[(wij, 0), (we, 10), (ons, 0), (onze, 0), (met...",10,"[(jij, 0), (je, 11), (jou, 0), (uw, 0), (u, 0)...",11,"[(jullie, 1), (u, 0), (met jullie, 0)]",1,"[(hij, 5), (hem, 4), (zijn, 0), (met hem, 0), ...",17,"[(zij, 0), (ze, 1), (hen, 0), (hun, 0), (met h...",1
1,maar liefst duizend euro krijgt van de overhei...,"[(ik, 26), (mij, 0), (me, 2), (mijn, 2), (met ...",30,"[(wij, 2), (we, 6), (ons, 2), (onze, 1), (met ...",11,"[(jij, 8), (je, 76), (jou, 5), (uw, 0), (u, 0)...",89,"[(jullie, 3), (u, 0), (met jullie, 0)]",3,"[(hij, 1), (hem, 1), (zijn, 0), (met hem, 0), ...",27,"[(zij, 1), (ze, 6), (hen, 0), (hun, 0), (met h...",7
2,we leven in een rare en moeilijke tijd. in de ...,"[(ik, 61), (mij, 3), (me, 3), (mijn, 5), (met ...",72,"[(wij, 0), (we, 31), (ons, 0), (onze, 0), (met...",31,"[(jij, 1), (je, 20), (jou, 0), (uw, 0), (u, 0)...",21,"[(jullie, 2), (u, 0), (met jullie, 0)]",2,"[(hij, 2), (hem, 0), (zijn, 1), (met hem, 0), ...",36,"[(zij, 0), (ze, 2), (hen, 0), (hun, 0), (met h...",2
3,valt ik op vrouwen? ik heb heel lang getwijfel...,"[(ik, 106), (mij, 18), (me, 6), (mijn, 17), (m...",147,"[(wij, 1), (we, 5), (ons, 1), (onze, 0), (met ...",7,"[(jij, 3), (je, 8), (jou, 2), (uw, 0), (u, 0),...",13,"[(jullie, 0), (u, 0), (met jullie, 0)]",0,"[(hij, 1), (hem, 0), (zijn, 1), (met hem, 0), ...",17,"[(zij, 1), (ze, 10), (hen, 0), (hun, 0), (met ...",11
4,"wie is daar? kom voor de houstour. oké, ik kom...","[(ik, 19), (mij, 6), (me, 0), (mijn, 19), (met...",44,"[(wij, 1), (we, 41), (ons, 2), (onze, 5), (met...",49,"[(jij, 3), (je, 23), (jou, 0), (uw, 1), (u, 0)...",27,"[(jullie, 5), (u, 0), (met jullie, 0)]",5,"[(hij, 0), (hem, 2), (zijn, 1), (met hem, 0), ...",22,"[(zij, 0), (ze, 3), (hen, 0), (hun, 1), (met h...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,"nou ben ik samen met felix, die jullie natuurl...","[(ik, 168), (mij, 10), (me, 6), (mijn, 19), (m...",203,"[(wij, 8), (we, 59), (ons, 3), (onze, 4), (met...",74,"[(jij, 7), (je, 40), (jou, 4), (uw, 0), (u, 0)...",51,"[(jullie, 14), (u, 0), (met jullie, 0)]",14,"[(hij, 29), (hem, 4), (zijn, 9), (met hem, 0),...",116,"[(zij, 3), (ze, 9), (hen, 0), (hun, 2), (met h...",14
116,artie is uitlogeren en NRP sfeer te zoeken hie...,"[(ik, 276), (mij, 26), (me, 8), (mijn, 35), (m...",345,"[(wij, 15), (we, 66), (ons, 6), (onze, 0), (me...",87,"[(jij, 16), (je, 63), (jou, 7), (uw, 0), (u, 0...",86,"[(jullie, 32), (u, 0), (met jullie, 0)]",32,"[(hij, 38), (hem, 14), (zijn, 7), (met hem, 0)...",172,"[(zij, 2), (ze, 23), (hen, 0), (hun, 3), (met ...",28
117,"ik heb die ingekocht, die, die, die heb ik wee...","[(ik, 196), (mij, 8), (me, 7), (mijn, 22), (me...",233,"[(wij, 8), (we, 60), (ons, 6), (onze, 3), (met...",77,"[(jij, 5), (je, 33), (jou, 3), (uw, 0), (u, 0)...",41,"[(jullie, 13), (u, 0), (met jullie, 0)]",13,"[(hij, 28), (hem, 14), (zijn, 9), (met hem, 0)...",121,"[(zij, 1), (ze, 11), (hen, 0), (hun, 0), (met ...",12
118,"ja, dus vandaar dat ik dit ook gewoon echt met...","[(ik, 310), (mij, 12), (me, 15), (mijn, 48), (...",385,"[(wij, 1), (we, 37), (ons, 1), (onze, 1), (met...",40,"[(jij, 7), (je, 77), (jou, 0), (uw, 0), (u, 1)...",85,"[(jullie, 15), (u, 1), (met jullie, 0)]",16,"[(hij, 9), (hem, 16), (zijn, 3), (met hem, 0),...",140,"[(zij, 4), (ze, 15), (hen, 0), (hun, 3), (met ...",22


File has been saved as: /Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Datasets/Dutch/Dutch_Pronouns_Dataset.csv
